In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def resolve_results_dir() -> Path:
    """Prefer ./results when the kernel cwd is `notebooks/`; also try repo-relative paths."""
    for candidate in (
        Path("results"),
        Path("notebooks/results"),
        Path.cwd() / "results",
    ):
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find a results folder. Run this notebook with cwd = `notebooks/` "
        "or create `./results` next to this notebook."
    )


def is_under_crypto_strategy(results: Path, fp: Path) -> bool:
    rel = fp.relative_to(results)
    return bool(rel.parts) and rel.parts[0].endswith("_crypto")


RESULTS = resolve_results_dir()
print(f"Results directory: {RESULTS}\n")

all_csv = sorted(RESULTS.rglob("*.csv"))
csv_files = [fp for fp in all_csv if is_under_crypto_strategy(RESULTS, fp)]
print(f"Found {len(csv_files)} crypto CSV file(s) (of {len(all_csv)} total).\n")

merics_parts: list[pd.DataFrame] = []
trade_inventory: list[dict] = []

for fp in csv_files:
    rel = fp.relative_to(RESULTS)
    parts = rel.parts
    strategy = symbol = timeframe = ""
    if len(parts) >= 4:
        strategy, symbol, timeframe = parts[0], parts[1], parts[2]
    elif len(parts) >= 2:
        strategy = parts[0]

    try:
        df = pd.read_csv(fp)
    except Exception as exc:
        print(f"[skip] {rel} — {exc}")
        continue

    low = fp.name.lower()
    if low == "merics.csv":
        extra = pd.DataFrame(
            {
                "strategy": [strategy] * len(df),
                "symbol": [symbol] * len(df),
                "timeframe": [timeframe] * len(df),
                "rel_path": [str(rel)] * len(df),
            }
        )
        merics_parts.append(pd.concat([extra.reset_index(drop=True), df.reset_index(drop=True)], axis=1))
    elif low == "trades.csv":
        trade_inventory.append(
            {
                "strategy": strategy,
                "symbol": symbol,
                "timeframe": timeframe,
                "rows": len(df),
                "rel_path": str(rel),
            }
        )

display(Markdown("### Combined metrics (`merics.csv`) — crypto only"))
if merics_parts:
    all_metrics = pd.concat(merics_parts, ignore_index=True)
    front = ["strategy", "symbol", "timeframe", "rel_path"]
    rest = [c for c in all_metrics.columns if c not in front]
    all_metrics = all_metrics[front + sorted(rest, key=str.lower)]
    display(
        all_metrics.sort_values(["strategy", "symbol", "timeframe"]).reset_index(drop=True)
    )
else:
    display(Markdown("_No `merics.csv` files found under `*_crypto/`._"))

display(Markdown("### Trades files (`trades.csv`) — crypto only"))
if trade_inventory:
    display(
        pd.DataFrame(trade_inventory).sort_values(
            ["strategy", "symbol", "timeframe"]
        ).reset_index(drop=True)
    )
else:
    display(Markdown("_No `trades.csv` files found under `*_crypto/`._"))

display(Markdown("### Every crypto CSV path (relative to results root)"))
for fp in csv_files:
    print(fp.relative_to(RESULTS))

Results directory: D:\bot\ema-1d trend-exchange\notebooks\results

Found 24 crypto CSV file(s) (of 28 total).



### Combined metrics (`merics.csv`) — crypto only

,strategy,symbol,timeframe,rel_path,avg_pnl,end_balance,max_drawdown_%,net_pnl,profit_factor,return_%,start_balance,trades,win_rate_%
0,strategy03_crypto,ADAUSDT,M5,strategy03_crypto\ADAUSDT\M5\merics.csv,2.97,11166.73,20.11,1166.73,1.053,11.67,10000.0,393,51.65
1,strategy03_crypto,BCHUSDT,M5,strategy03_crypto\BCHUSDT\M5\merics.csv,13.86,17066.24,8.41,7066.24,1.239,70.66,10000.0,510,55.49
2,strategy03_crypto,BNBUSDT,M5,strategy03_crypto\BNBUSDT\M5\merics.csv,18.49,21166.28,9.37,11166.28,1.270,111.66,10000.0,604,56.46
3,strategy03_crypto,BTCUSDT,M5,strategy03_crypto\BTCUSDT\M5\merics.csv,22.86,22323.98,8.40,12323.98,1.372,123.24,10000.0,539,57.70
4,strategy03_crypto,ETHUSDT,M5,strategy03_crypto\ETHUSDT\M5\merics.csv,28.32,24696.58,8.38,14696.58,1.426,146.97,10000.0,519,58.96
5,strategy03_crypto,XRPUSDT,M5,strategy03_crypto\XRPUSDT\M5\merics.csv,25.55,23258.32,8.47,13258.32,1.388,132.58,10000.0,519,58.38


### Trades files (`trades.csv`) — crypto only

,strategy,symbol,timeframe,rows,rel_path
0,strategy03_crypto,ADAUSDT,M5,393,strategy03_crypto\ADAUSDT\M5\trades.csv
1,strategy03_crypto,BCHUSDT,M5,510,strategy03_crypto\BCHUSDT\M5\trades.csv
2,strategy03_crypto,BNBUSDT,M5,604,strategy03_crypto\BNBUSDT\M5\trades.csv
3,strategy03_crypto,BTCUSDT,M5,539,strategy03_crypto\BTCUSDT\M5\trades.csv
4,strategy03_crypto,ETHUSDT,M5,519,strategy03_crypto\ETHUSDT\M5\trades.csv
5,strategy03_crypto,XRPUSDT,M5,519,strategy03_crypto\XRPUSDT\M5\trades.csv


### Every crypto CSV path (relative to results root)

strategy03_crypto\ADAUSDT\M5\entry_fills.csv
strategy03_crypto\ADAUSDT\M5\entry_signals.csv
strategy03_crypto\ADAUSDT\M5\merics.csv
strategy03_crypto\ADAUSDT\M5\trades.csv
strategy03_crypto\BCHUSDT\M5\entry_fills.csv
strategy03_crypto\BCHUSDT\M5\entry_signals.csv
strategy03_crypto\BCHUSDT\M5\merics.csv
strategy03_crypto\BCHUSDT\M5\trades.csv
strategy03_crypto\BNBUSDT\M5\entry_fills.csv
strategy03_crypto\BNBUSDT\M5\entry_signals.csv
strategy03_crypto\BNBUSDT\M5\merics.csv
strategy03_crypto\BNBUSDT\M5\trades.csv
strategy03_crypto\BTCUSDT\M5\entry_fills.csv
strategy03_crypto\BTCUSDT\M5\entry_signals.csv
strategy03_crypto\BTCUSDT\M5\merics.csv
strategy03_crypto\BTCUSDT\M5\trades.csv
strategy03_crypto\ETHUSDT\M5\entry_fills.csv
strategy03_crypto\ETHUSDT\M5\entry_signals.csv
strategy03_crypto\ETHUSDT\M5\merics.csv
strategy03_crypto\ETHUSDT\M5\trades.csv
strategy03_crypto\XRPUSDT\M5\entry_fills.csv
strategy03_crypto\XRPUSDT\M5\entry_signals.csv
strategy03_crypto\XRPUSDT\M5\merics.csv
strategy